# Coffee Chain Daily Sales Forecast with Chronos

Notebook นี้ทำสำหรับรันบน Google Colab เพื่อทำนายยอดขายรายวันระดับ `store_id x category` ด้วย Chronos-2 และวัดผลด้วย MAE

แนวคิดหลัก:
- รวมยอดขายจาก `INVENTORY.csv` เป็นยอดขายรายวันของแต่ละร้านและหมวดสินค้า
- สร้าง time series หนึ่งเส้นต่อ `store_id + category`
- ใช้ Chronos-2 พยากรณ์ยอดขายรายวันในอนาคต
- แปลงผลรายวันเป็นคำตอบ `1d`, `7d`, `1m` ตาม `sample_submission_with_id.csv`
- ใช้ MAE ประเมินผลบน validation split ก่อนสร้าง submission

อ้างอิง API ปัจจุบันของ Chronos: https://github.com/amazon-science/chronos-forecasting

## 1. ติดตั้งไลบรารี

แนะนำให้เลือก Runtime เป็น GPU ใน Colab: `Runtime > Change runtime type > T4 GPU`

In [ ]:
!pip -q install chronos-forecasting pandas pyarrow accelerate transformers matplotlib scikit-learn

## 2. โหลดข้อมูล

อัปโหลดโฟลเดอร์ข้อมูลขึ้น Colab หรือ Google Drive แล้วแก้ `DATA_DIR` ให้ชี้ไปยังโฟลเดอร์ที่มี `train/`, `test/`, และ `sample_submission_with_id.csv`

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')

# ตัวอย่าง Colab Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/super-ai-engineer-season-6-coffee-chain-hackathon')

DATA_DIR = Path('/content/super-ai-engineer-season-6-coffee-chain-hackathon')

train_dir = DATA_DIR / 'train'
test_dir = DATA_DIR / 'test'
sample_path = DATA_DIR / 'sample_submission_with_id.csv'

inventory = pd.read_csv(train_dir / 'INVENTORY.csv', parse_dates=['date'])
product = pd.read_csv(train_dir / 'PRODUCT.csv')
store = pd.read_csv(train_dir / 'STORE.csv', parse_dates=['opened_date'])
date_dim = pd.read_csv(test_dir / 'DATE_DIM.csv', parse_dates=['date'])
local_event = pd.read_csv(test_dir / 'LOCAL_EVENT.csv', parse_dates=['date'])
promotion = pd.read_csv(test_dir / 'PROMOTION.csv', parse_dates=['start_date', 'end_date'])
sample_submission = pd.read_csv(sample_path)

print('inventory:', inventory.shape)
print('product:', product.shape)
print('store:', store.shape)
print('sample:', sample_submission.shape)

## 3. อธิบายฟีเจอร์ที่ใช้

| กลุ่ม | ฟีเจอร์ | ความหมาย / เหตุผลที่ใช้ |
|---|---|---|
| Target history | `target` | ยอดขาย `units_sold` รายวันของร้านและหมวดสินค้า เป็นสัญญาณหลักที่ Chronos ใช้เรียนรู้ pattern เช่น trend, weekly seasonality และช่วงขายดี/ขายตก |
| Series id | `item_id` | ระบุ time series แต่ละเส้นในรูป `store_id__category` เช่น ร้าน 1 หมวด Coffee เพื่อให้โมเดลพยากรณ์แยกกันตามพฤติกรรมของแต่ละร้านและหมวด |
| Calendar | `day_of_week`, `week_number`, `month`, `quarter`, `year` | จับ seasonality ตามวันในสัปดาห์ เดือน ไตรมาส และปี เช่น เสาร์อาทิตย์หรือปลายปีขายต่างจากวันธรรมดา |
| Calendar flags | `is_weekend`, `is_holiday`, `is_school_break`, `is_payday`, `is_rainy_season` | เหตุการณ์ตามปฏิทินที่มีผลต่อ traffic และกำลังซื้อ เช่น วันหยุด เงินเดือนออก ช่วงปิดเทอม หรือหน้าฝน |
| Store profile | `seating_capacity`, `staff_count`, `has_drive_through`, `operating_hours`, `store_age_days` | บอก capacity และลักษณะการให้บริการของร้าน ร้านใหญ่/เปิดนาน/มี drive-through อาจมียอดขายต่างกัน |
| Store location | `neighborhood_type_*` | one-hot ของย่านร้าน เช่น university, tourist, hospital ช่วยจับ demand ต่างกันตามบริบทพื้นที่ |
| Promotion | `promo_count`, `avg_discount_pct`, `max_discount_pct`, `email_sent_count`, `social_campaign_count` | สรุปโปรโมชันที่ active ในวันนั้นระดับร้านและหมวดสินค้า ส่วนลดและแคมเปญมักทำให้ยอดขายเปลี่ยน |
| Local event | `event_count`, `event_type_*` | จำนวนและประเภท event ใกล้ร้านในวันนั้น เช่น food festival หรือ cultural event ที่อาจเพิ่ม traffic |
| Category | `category_*` | one-hot ของหมวดสินค้า เพื่อให้ covariate บอกชนิดสินค้า แม้ `item_id` จะมี category อยู่แล้วก็ตาม |

หมายเหตุ: Chronos-2 รองรับ covariates ผ่าน `future_df` แต่ถ้าใช้ Chronos-Bolt/T5 แบบดั้งเดิมจะเป็น univariate เป็นหลัก ดังนั้น notebook นี้เลือก `Chronos2Pipeline` เพื่อให้ใช้ฟีเจอร์อนาคตที่รู้ล่วงหน้าได้

## 4. เตรียม target รายวันระดับร้าน-หมวดสินค้า

In [ ]:
def parse_submission_id(value):
    store_id, category, date, horizon = value.rsplit('_', 3)
    return int(store_id), category, pd.Timestamp(date), horizon

id_parts = sample_submission['id'].map(parse_submission_id)
id_frame = pd.DataFrame(id_parts.tolist(), columns=['store_id', 'category', 'date', 'horizon'])

forecast_start = id_frame['date'].min()
forecast_end = id_frame['date'].max()
max_extra_days = 30  # ใช้สำหรับคำนวณ 1m เป็นยอดรวม 30 วันข้างหน้า
final_prediction_end = forecast_end + pd.Timedelta(days=max_extra_days)

categories = sorted(id_frame['category'].unique())
stores = sorted(id_frame['store_id'].unique())

sales_daily = (
    inventory.merge(product[['product_id', 'category']], on='product_id', how='left')
    .groupby(['store_id', 'category', 'date'], as_index=False)['units_sold'].sum()
)

full_dates = pd.date_range(sales_daily['date'].min(), final_prediction_end, freq='D')
grid = pd.MultiIndex.from_product(
    [stores, categories, full_dates],
    names=['store_id', 'category', 'timestamp']
).to_frame(index=False)

target_df = grid.merge(
    sales_daily.rename(columns={'date': 'timestamp', 'units_sold': 'target'}),
    on=['store_id', 'category', 'timestamp'],
    how='left'
)

# เติม 0 เฉพาะอดีตที่ไม่มีรายการขาย ส่วนอนาคตปล่อยเป็น NaN เพื่อให้โมเดลพยากรณ์
last_train_date = sales_daily['date'].max()
past_mask = target_df['timestamp'] <= last_train_date
target_df.loc[past_mask, 'target'] = target_df.loc[past_mask, 'target'].fillna(0)
target_df['item_id'] = target_df['store_id'].astype(str) + '__' + target_df['category']

print('train date range:', sales_daily['date'].min().date(), 'to', last_train_date.date())
print('forecast date range:', forecast_start.date(), 'to', forecast_end.date())
print('series:', target_df['item_id'].nunique())
target_df.head()

## 5. สร้างฟีเจอร์อนาคตที่รู้ล่วงหน้า

In [ ]:
def make_calendar_features(dates, date_dim):
    cal = pd.DataFrame({'timestamp': pd.to_datetime(dates)})
    base = date_dim.rename(columns={'date': 'timestamp'}).copy()
    cal = cal.merge(base, on='timestamp', how='left')

    missing = cal['day_of_week'].isna()
    if missing.any():
        ts = cal.loc[missing, 'timestamp']
        cal.loc[missing, 'day_of_week'] = ts.dt.day_name()
        cal.loc[missing, 'week_number'] = ts.dt.isocalendar().week.astype(int).to_numpy()
        cal.loc[missing, 'month'] = ts.dt.month
        cal.loc[missing, 'quarter'] = ts.dt.quarter
        cal.loc[missing, 'year'] = ts.dt.year
        cal.loc[missing, 'is_weekend'] = ts.dt.dayofweek >= 5
        cal.loc[missing, 'is_holiday'] = False
        cal.loc[missing, 'holiday_name'] = ''
        cal.loc[missing, 'is_school_break'] = False
        cal.loc[missing, 'is_payday'] = ts.dt.day.isin([25, 30, 31])
        cal.loc[missing, 'is_rainy_season'] = ts.dt.month.isin([5, 6, 7, 8, 9, 10])

    bool_cols = ['is_weekend', 'is_holiday', 'is_school_break', 'is_payday', 'is_rainy_season']
    for col in bool_cols:
        cal[col] = cal[col].fillna(False).astype(bool).astype(int)

    cal['holiday_name'] = cal['holiday_name'].fillna('none').replace('', 'none')
    cal = pd.get_dummies(cal, columns=['day_of_week', 'holiday_name'], drop_first=False, dtype=int)
    return cal

def make_store_features(store):
    out = store.copy()
    out['open_time'] = pd.to_datetime(out['open_time'], format='%H:%M')
    out['close_time'] = pd.to_datetime(out['close_time'], format='%H:%M')
    out['operating_hours'] = (out['close_time'] - out['open_time']).dt.total_seconds() / 3600
    out['store_age_days'] = (forecast_start - out['opened_date']).dt.days.clip(lower=0)
    out['has_drive_through'] = out['has_drive_through'].astype(bool).astype(int)
    out = out[['store_id', 'neighborhood_type', 'seating_capacity', 'has_drive_through', 'staff_count', 'operating_hours', 'store_age_days']]
    out = pd.get_dummies(out, columns=['neighborhood_type'], drop_first=False, dtype=int)
    return out

def make_event_features(local_event):
    ev = local_event.rename(columns={'date': 'timestamp'}).copy()
    ev['event_count'] = 1
    ev_type = pd.get_dummies(ev['event_type'], prefix='event_type', dtype=int)
    ev = pd.concat([ev[['store_id', 'timestamp', 'event_count']], ev_type], axis=1)
    return ev.groupby(['store_id', 'timestamp'], as_index=False).sum()

def make_promo_features(promotion, product):
    promo = promotion.merge(product[['product_id', 'category']], on='product_id', how='left')
    rows = []
    for row in promo.itertuples(index=False):
        for day in pd.date_range(row.start_date, row.end_date, freq='D'):
            rows.append({
                'store_id': row.store_id,
                'category': row.category,
                'timestamp': day,
                'promo_count': 1,
                'discount_pct': row.discount_pct,
                'email_sent_count': int(bool(row.email_sent)),
                'social_campaign_count': int(bool(row.social_campaign)),
            })
    promo_daily = pd.DataFrame(rows)
    if promo_daily.empty:
        return pd.DataFrame(columns=['store_id', 'category', 'timestamp', 'promo_count', 'avg_discount_pct', 'max_discount_pct', 'email_sent_count', 'social_campaign_count'])
    return (
        promo_daily.groupby(['store_id', 'category', 'timestamp'], as_index=False)
        .agg(
            promo_count=('promo_count', 'sum'),
            avg_discount_pct=('discount_pct', 'mean'),
            max_discount_pct=('discount_pct', 'max'),
            email_sent_count=('email_sent_count', 'sum'),
            social_campaign_count=('social_campaign_count', 'sum'),
        )
    )

calendar_features = make_calendar_features(full_dates, date_dim)
store_features = make_store_features(store)
event_features = make_event_features(local_event)
promo_features = make_promo_features(promotion, product)

feature_df = (
    grid.merge(calendar_features, on='timestamp', how='left')
    .merge(store_features, on='store_id', how='left')
    .merge(event_features, on=['store_id', 'timestamp'], how='left')
    .merge(promo_features, on=['store_id', 'category', 'timestamp'], how='left')
)

category_dummies = pd.get_dummies(feature_df['category'], prefix='category', drop_first=False, dtype=int)
feature_df = pd.concat([feature_df, category_dummies], axis=1)
feature_df = feature_df.fillna(0)
model_df = target_df[['item_id', 'timestamp', 'target', 'store_id', 'category']].merge(
    feature_df,
    on=['store_id', 'category', 'timestamp'],
    how='left'
)
model_df = model_df.drop(columns=['store_id', 'category'])

feature_cols = [c for c in model_df.columns if c not in ['item_id', 'timestamp', 'target']]
model_df[feature_cols] = model_df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

print('model_df:', model_df.shape)
print('feature count:', len(feature_cols))
model_df.head()

## 6. ฟังก์ชันพยากรณ์ด้วย Chronos-2 และคำนวณ MAE

In [ ]:
from chronos import Chronos2Pipeline

MODEL_ID = 'autogluon/chronos-2-small'  # เปลี่ยนเป็น 'amazon/chronos-2' ได้ถ้า GPU/เวลาเพียงพอ
device_map = 'cuda' if torch.cuda.is_available() else 'cpu'

pipeline = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map=device_map)
print('loaded:', MODEL_ID, 'on', device_map)

def chronos_predict(model_df, cutoff_date, prediction_length, quantile_levels=(0.1, 0.5, 0.9)):
    cutoff_date = pd.Timestamp(cutoff_date)
    context_df = model_df[model_df['timestamp'] <= cutoff_date].copy()
    future_df = model_df[
        (model_df['timestamp'] > cutoff_date)
        & (model_df['timestamp'] <= cutoff_date + pd.Timedelta(days=prediction_length))
    ].drop(columns=['target']).copy()

    pred_df = pipeline.predict_df(
        context_df,
        future_df=future_df,
        prediction_length=prediction_length,
        quantile_levels=list(quantile_levels),
        id_column='item_id',
        timestamp_column='timestamp',
        target='target',
    )

    pred_col = 'predictions' if 'predictions' in pred_df.columns else '0.5'
    pred_df = pred_df.rename(columns={pred_col: 'prediction'})
    pred_df['prediction'] = pred_df['prediction'].clip(lower=0)
    return pred_df

def add_rolling_horizon(df, value_col):
    out = df.sort_values(['item_id', 'timestamp']).copy()
    out['pred_1d'] = out[value_col]
    out['pred_7d'] = out.groupby('item_id')[value_col].transform(
        lambda s: s.iloc[::-1].rolling(7, min_periods=7).sum().iloc[::-1]
    )
    # คำนวณ 1m เป็น rolling sum 30 วันข้างหน้า รวมวันปัจจุบัน
    out['pred_1m'] = out.groupby('item_id')[value_col].transform(
        lambda s: s.iloc[::-1].rolling(30, min_periods=30).sum().iloc[::-1]
    )
    return out

## 7. Validation ด้วย MAE

ใช้ข้อมูลถึง `2024-08-31` เพื่อพยากรณ์ `2024-09-01` ถึง `2024-10-31` แล้วคำนวณ MAE รายวัน (`1d`) และ MAE ของ rolling horizon (`7d`, `1m`) เฉพาะวันที่มี ground truth ครบหน้าต่าง

In [ ]:
val_cutoff = pd.Timestamp('2024-08-31')
val_prediction_length = 61

val_pred = chronos_predict(model_df, val_cutoff, val_prediction_length)
actual = model_df[(model_df['timestamp'] > val_cutoff) & (model_df['timestamp'] <= val_cutoff + pd.Timedelta(days=val_prediction_length))][['item_id', 'timestamp', 'target']]
val_eval = val_pred[['item_id', 'timestamp', 'prediction']].merge(actual, on=['item_id', 'timestamp'], how='inner')

mae_1d = mean_absolute_error(val_eval['target'], val_eval['prediction'])
print(f'Validation MAE 1d: {mae_1d:,.4f}')

pred_roll = add_rolling_horizon(val_eval[['item_id', 'timestamp', 'prediction']].rename(columns={'prediction': 'value'}), 'value')
true_roll = add_rolling_horizon(val_eval[['item_id', 'timestamp', 'target']].rename(columns={'target': 'value'}), 'value')

roll_eval = pred_roll[['item_id', 'timestamp', 'pred_1d', 'pred_7d', 'pred_1m']].merge(
    true_roll[['item_id', 'timestamp', 'pred_1d', 'pred_7d', 'pred_1m']],
    on=['item_id', 'timestamp'],
    suffixes=('_pred', '_true')
)

for h in ['1d', '7d', '1m']:
    tmp = roll_eval.dropna(subset=[f'pred_{h}_pred', f'pred_{h}_true'])
    print(f'Validation MAE {h}: {mean_absolute_error(tmp[f"pred_{h}_true"], tmp[f"pred_{h}_pred"]):,.4f}  n={len(tmp):,}')

## 8. Train ด้วยข้อมูลทั้งหมดและสร้าง submission

In [ ]:
final_prediction_length = (final_prediction_end - last_train_date).days
final_pred = chronos_predict(model_df, last_train_date, final_prediction_length)

daily_pred = final_pred[['item_id', 'timestamp', 'prediction']].rename(columns={'prediction': 'value'})
daily_roll = add_rolling_horizon(daily_pred, 'value')

pred_map = daily_roll.set_index(['item_id', 'timestamp'])[['pred_1d', 'pred_7d', 'pred_1m']]

def lookup_prediction(row):
    store_id, category, date, horizon = parse_submission_id(row['id'])
    item_id = f'{store_id}__{category}'
    col = f'pred_{horizon}'
    value = pred_map.loc[(item_id, date), col]
    return max(0.0, float(value))

submission = sample_submission.copy()
submission['units_sold_predicted'] = submission.apply(lookup_prediction, axis=1)
submission['units_sold_predicted'] = submission['units_sold_predicted'].fillna(0).clip(lower=0)

out_path = DATA_DIR / 'submission_chronos.csv'
submission.to_csv(out_path, index=False)
print('saved:', out_path)
submission.head(10)

## 9. วิธีปรับให้คะแนนดีขึ้น

- ถ้า Colab GPU ยังไหว ให้เปลี่ยน `MODEL_ID` เป็น `amazon/chronos-2` เพื่อใช้โมเดลใหญ่ขึ้น
- ทดลองนิยาม `1m` เป็น 31 วัน หรือเดือนปฏิทิน ถ้ากติกาการแข่งขันระบุไว้ต่างจาก rolling 30 วัน
- ใช้ `TRANSACTION + ORDER` เพื่อสร้าง target อีกชุดแล้วเทียบ MAE กับ target จาก `INVENTORY`
- เพิ่ม post-processing เช่น clip ตาม percentile ของยอดขายแต่ละ `item_id` เพื่อลด outlier
- ทำ ensemble ระหว่าง Chronos กับ baseline แบบ seasonal naive หรือ LightGBM แล้วเลือกด้วย validation MAE